# AN6 — Recorded VLE Activity Patterns

**Status:** Prepared; no SQL outputs are embedded. Run cells individually, not Run All. This notebook describes recorded platform activity, not learning quality.


## 1. Environment and shared query runner

Canonical SQL remains in `sql/04_analytics/`. Results are retained in `query_results`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd()
while ROOT!=ROOT.parent and not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT)) if str(ROOT) not in sys.path else None
from src.analytics.db import run_query
query_results={}
EXPECTED={'AN6.01':3,'AN6.02':1,'AN6.03':1,'AN6.04':1}
HIGH_COST_TIMEOUT_MS=600_000


## 2. AN0.01 — Read-only preflight

Confirm the intended database and read-only settings. Stop if any check fails.


In [ ]:
preflight=run_query('AN0.01',max_rows=10,statement_timeout_ms=120_000)
assert preflight['result_set_count']==1 and not preflight['data'].empty
row=preflight['data'].iloc[0]
checks={'database':row['database_name']=='oulad_analytics','transaction_read_only':str(row['transaction_read_only']).lower()=='on','default_transaction_read_only':str(row['default_transaction_read_only']).lower()=='on'}
print(checks)
assert all(checks.values()), 'Stop: read-only preflight failed.'


## 3. AN6.01 — Readiness and metadata

**Question:** Are course windows valid, are VLE resource IDs unique, and are registration/activity keys complete? Review all three outputs; failures block downstream queries.


In [ ]:
r=run_query('AN6.01',max_rows=500,statement_timeout_ms=120_000)
assert r['result_set_count']==EXPECTED['AN6.01']
query_results['AN6.01']=r
for item in r['result_sets']: display(item['data'])
course,meta,keys=[x['data'] for x in r['result_sets']]
assert course['valid_course_window'].fillna(False).all() and (course['course_key_rows']==1).all()
assert int(meta.iloc[0]['excess_rows_over_distinct_id_site'])==0 and int(meta.iloc[0]['null_id_site_rows'])==0
assert int(keys.iloc[0]['duplicate_registration_keys'])==0 and int(keys.iloc[0]['activity_rows_with_null_key_component'])==0
print('Readiness passed; metadata missingness is descriptive:',meta[['missing_activity_type_rows','missing_week_from_rows','missing_week_to_rows']])


## 4. AN6.02 — In-course coverage and student-presentation distributions

**Question:** What share of registered student-presentations has in-course activity? How are zero-inclusive active days and resource breadth distributed? First activity day is conditional on activity being recorded.


In [ ]:
r=run_query('AN6.02',max_rows=500,statement_timeout_ms=HIGH_COST_TIMEOUT_MS)
assert r['result_set_count']==EXPECTED['AN6.02']
query_results['AN6.02']=r
an602=r['result_sets'][0]['data']
display(an602)


## 5. AN6.03 — Temporal patterns

**Question:** How do activity rows distribute across PRE/IN/POST and fixed seven-day bins? These bins are not academic weeks.


In [ ]:
r=run_query('AN6.03',max_rows=500,statement_timeout_ms=HIGH_COST_TIMEOUT_MS)
assert r['result_set_count']==EXPECTED['AN6.03']
query_results['AN6.03']=r
display(r['result_sets'][0]['data'])


## 6. AN6.04 — Activity-type composition

**Question:** Which resource types contribute to in-course resource breadth and registered student-presentation reach? Missing and unmatched metadata remain explicit.


In [ ]:
r=run_query('AN6.04',max_rows=500,statement_timeout_ms=HIGH_COST_TIMEOUT_MS)
assert r['result_set_count']==EXPECTED['AN6.04']
query_results['AN6.04']=r
display(r['result_sets'][0]['data'])


## 7. Descriptive visual summary

Charts are created only from returned SQL frames; no pre-execution figures or findings are embedded.


In [ ]:
if 'AN6.02' in query_results:
    frame=query_results['AN6.02']['result_sets'][0]['data']
    cov=frame[frame.output_section=='COVERAGE'].copy()
    if not cov.empty:
        cov['presentation']=cov.code_module.astype(str)+' '+cov.code_presentation.astype(str)
        ax=cov.plot(x='presentation',y='pct_or_mean',kind='bar',legend=False,figsize=(12,5),title='Recorded in-course activity coverage by presentation')
        ax.set_ylabel('Registered student-presentations with in-course activity (%)')
        plt.tight_layout(); plt.show()
    display(frame[frame.output_section=='DISTRIBUTION'])


## 8. Review checklist

- Verify result-set counts and truncation/runtime metadata.
- Reconcile registration denominators and PRE/IN/POST activity rows.
- Review missing/unmatched VLE metadata.
- Do not interpret missing VLE rows as absence of learning or negative course-relative days as pre-registration activity.
- No composite score, threshold, outcome claim, or predictive feature is approved in AN6.
